In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np

from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.utils import to_categorical

with open('text', 'r', encoding='utf-8') as f:
    texts = f.read()
    texts = texts.replace('\ufeff', '')  # убираем первый невидимый символ

maxWordsCount = 1000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts([texts])

dist = list(tokenizer.word_counts.items())
print(dist[:10])

data = tokenizer.texts_to_sequences([texts])
# res = to_categorical(data[0], num_classes=maxWordsCount)
# print(res.shape)
res = np.array( data[0] )

inp_words = 3
n = res.shape[0] - inp_words

X = np.array([res[i:i + inp_words] for i in range(n)])
Y = to_categorical(res[inp_words:], num_classes=maxWordsCount)

model = Sequential()
model.add(Embedding(maxWordsCount, 256, input_length = inp_words))
# model.add(SimpleRNN(128, activation='tanh'))
model.add(SimpleRNN (128, activation= 'tanh', return_sequences=True))  #(batch_size, timesteps, units)
model.add(SimpleRNN (64, activation= 'tanh'))
model.add(Dense(maxWordsCount, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=50)


def buildPhrase(texts, str_len=20):
    res = texts
    data = tokenizer.texts_to_sequences([texts])[0]
    for i in range(str_len):
        # x = to_categorical(data[i: i + inp_words], num_classes=maxWordsCount)  # преобразуем в One-Hot-encoding
        # inp = x.reshape(1, inp_words, maxWordsCount)
        x = data[i: i + inp_words]
        inp = np.expand_dims(x, axis=0)

        pred = model.predict(inp)
        indx = pred.argmax(axis=1)[0]
        data.append(indx)

        res += " " + tokenizer.index_word[indx]  # дописываем строку

    return res


res = buildPhrase("позитив добавляет годы")
print(res)

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np

from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.utils import to_categorical

with open('text', 'r', encoding='utf-8') as f:
    texts = f.read()
    texts = texts.replace('\ufeff', '')  # убираем первый невидимый символ

maxWordsCount = 1000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts([texts])

dist = list(tokenizer.word_counts.items())
print(dist[:10])

data = tokenizer.texts_to_sequences([texts])
# res = to_categorical(data[0], num_classes=maxWordsCount)
# print(res.shape)
res = np.array( data[0] )

inp_words = 3
n = res.shape[0] - inp_words

X = np.array([res[i:i + inp_words] for i in range(n)])
Y = to_categorical(res[inp_words:], num_classes=maxWordsCount)

model = Sequential()
model.add(Embedding(maxWordsCount, 256, input_length = inp_words))
# model.add(SimpleRNN(128, activation='tanh'))  bunun əvəzinə altdakı
model.add(SimpleRNN (128, activation= 'tanh', return_sequences=True))  #(batch_size, timesteps, units)
# 1-ci RNN qatı: 128 neyron, tanh aktivasiyası
# return_sequences=True: qat hər zaman addımı (timestep) üçün çıxış qaytarır,
# yalnız sonuncunu yox. Çıxışın forması: (batch_size, timesteps, units) = (batch, 3, 128)
# Bu lazımdır ki, növbəti RNN qatı ardıcıllıq (sequence) alsın.
model.add(SimpleRNN (64, activation= 'tanh'))
# 2-ci RNN qatı: 64 neyron, tanh aktivasiyası
# return_sequences verilməyib (default False): yalnız SON addımın çıxışını qaytarır.
# Çıxışın forması: (batch_size, units) = (batch, 64)
# Dense qat yalnız bir vektor gözləyir, ona görə sonuncu RNN-də ardıcıllıq lazım deyil.
model.add(Dense(maxWordsCount, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=50)


def buildPhrase(texts, str_len=20):
    res = texts
    data = tokenizer.texts_to_sequences([texts])[0]
    for i in range(str_len):
        # x = to_categorical(data[i: i + inp_words], num_classes=maxWordsCount)  # преобразуем в One-Hot-encoding
        # inp = x.reshape(1, inp_words, maxWordsCount)
        x = data[i: i + inp_words]
        inp = np.expand_dims(x, axis=0)

        pred = model.predict(inp)
        indx = pred.argmax(axis=1)[0]
        data.append(indx)

        res += " " + tokenizer.index_word[indx]  # дописываем строку

    return res


res = buildPhrase("позитив добавляет годы")
print(res)

## Geniş İzah

Bu iki sətir **Deep RNN** (dərin rekurrent şəbəkə) qurur: iki SimpleRNN qatı üst-üstə yığılıb. Kommentlə izahı belədir:

```python
# 1-ci RNN qatı: 128 neyron, tanh aktivasiyası
# return_sequences=True: qat hər zaman addımı (timestep) üçün çıxış qaytarır,
# yalnız sonuncunu yox. Çıxışın forması: (batch_size, timesteps, units) = (batch, 3, 128)
# Bu lazımdır ki, növbəti RNN qatı ardıcıllıq (sequence) alsın.
model.add(SimpleRNN(128, activation='tanh', return_sequences=True))

# 2-ci RNN qatı: 64 neyron, tanh aktivasiyası
# return_sequences verilməyib (default False): yalnız SON addımın çıxışını qaytarır.
# Çıxışın forması: (batch_size, units) = (batch, 64)
# Dense qat yalnız bir vektor gözləyir, ona görə sonuncu RNN-də ardıcıllıq lazım deyil.
model.add(SimpleRNN(64, activation='tanh'))
```

## Niyə return_sequences=True lazımdır?

Embedding qatından sonra giriş forması `(batch, 3, 256)` olur (3 söz, hər biri 256 ölçülü vektor).

Əgər 1-ci qatda `return_sequences=True` olmasaydı, o yalnız son addımın nəticəsini verərdi: `(batch, 128)`. Bu 2-ci SimpleRNN-ə verilsəydi, xəta çıxardı, çünki RNN qatı 3 ölçülü giriş `(batch, timesteps, features)` gözləyir.

## Formaların axını

```
Embedding           → (batch, 3, 256)
SimpleRNN(128, return_sequences=True) → (batch, 3, 128)   # 3 addımın hamısı
SimpleRNN(64)       → (batch, 64)                         # yalnız sonuncu addım
Dense(1000, softmax) → (batch, 1000)                      # hər söz üçün ehtimal
```

## Nə dəyişir (əvvəlki tək qatlı versiyadan)

- Əvvəl: bir RNN (128) → Dense
- İndi: RNN (128) → RNN (64) → Dense

1-ci qat sözlər arasında aşağı səviyyəli əlaqələri öyrənir, 2-ci qat onun çıxışı üzərində daha mücərrəd əlaqələri öyrənir. Buna görə şəbəkə "dərin" adlanır.

## Qeyd

`SimpleRNN` uzun ardıcıllıqlarda zəifdir (vanishing gradient), amma sənin ardıcıllığın cəmi 3 sözdür, ona görə burada problem yaratmır. Daha güclü variant üçün `SimpleRNN` əvəzinə `LSTM` və ya `GRU` yazmaq kifayətdir, `return_sequences` məntiqi eyni qalır.